# Tutorial 18: Multi-Agent Consensus and Cross-Document Consistency

**Conceptual deep-dive:** `gmsh_testing_ext/chapters/ch18.tex` — Multi-Agent Consensus and Cross-Document Consistency
**Monograph notebook:** `gmsh_testing_ext/notebooks/ch18_multi_agent_cross_doc.ipynb`
**Companion guide section:** `knowlytix/harness/testing/USER_GUIDE.md` §21
**Prereqs:** NB 02 (Primitives), NB 14 (Environment Sim), NB 15 (Oracles)

**What you'll do.** Load three bank model-risk documents that disagree about specific facts (`ch18_doc_q1`, `q2`, `q3` — three quarterly reports for the same model inventory) into three GMS stores. Use `CrossDocumentAnalyzer` to find shared entities and detect conflicts between the documents. Use `ConsensusOracle.evaluate_consensus(entity)` to ask "what do all three sources agree about for MDL-001?" This is the multi-source verification surface a bank deployment needs when its agent reads model documentation from multiple business lines.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _shared import LAB_STORES_PATH, seed_all, REPO_ROOT
from knowlytix.harness.testing import ConsensusOracle, CrossDocumentAnalyzer
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
seed_all()
import os
os.chdir(REPO_ROOT)
print("imports OK")


## 2. Concept

The harness models cross-document consistency as **multi-store geometry**: each document becomes its own `GMSExpertStore`; an entity that appears in multiple stores becomes a multi-source claim about the same real-world thing.

- `CrossDocumentAnalyzer.find_shared_entities()` lists entities present in 2+ stores
- `.find_conflicts()` flags entities whose attribute values disagree across stores (different `tension_energy` between corresponding embeddings)
- `ConsensusOracle.evaluate_consensus(entity)` returns an `InteractionResult` with the consensus verdict for that entity across all loaded stores

`ch18_doc_q1`, `ch18_doc_q2`, `ch18_doc_q3` are three quarterly model-risk reports that overlap on MDL-001 through MDL-005 but disagree on some metrics (statuses, AUCs) — a realistic stand-in for cross-business-line reporting drift.

## 3. Load the Three Quarterly Stores

In [ ]:
import contextlib
import io
buf = io.StringIO()

stores = {}
for tag in ["q1", "q2", "q3"]:
    path = str(LAB_STORES_PATH / f"ch18_doc_{tag}")
    s = GMSExpertStore(DocGMSConfig(store_path=path))
    with contextlib.redirect_stdout(buf):
        s.load()
    stores[tag] = s
    stats = s.stats()
    print(f"  {tag}: {stats['entities']} entities, {stats['relations']} relations, "
          f"{stats['triples']} triples, {stats['enm_entries']} ENM")


## 4. Find Shared Entities Across Documents

In [ ]:
analyzer = CrossDocumentAnalyzer(stores=stores)
shared = analyzer.find_shared_entities()
print(f"Total shared entities across q1/q2/q3: {len(shared)}")
print()
sample = list(shared)[:10] if not isinstance(shared, dict) else list(shared.items())[:10]
print("First 10 shared entities:")
for item in sample:
    print(f"  {item}")

## 5. Detect Cross-Document Conflicts

In [ ]:
conflicts = analyzer.find_conflicts()
print(f"Detected conflicts: {len(conflicts)}")
print()
for c in conflicts[:10]:
    if hasattr(c, "entity"):
        print(f"  entity={c.entity} tension={getattr(c, 'tension_score', 'n/a')} "
              f"stores={getattr(c, 'stores', None)}")
    else:
        print(f"  {c}")

## 6. Consensus on MDL-001 Across All Three Stores

In [ ]:
oracle = ConsensusOracle(stores=stores)
for entity in ["mdl-001", "mdl-002", "mdl-005"]:
    result = oracle.evaluate_consensus(entity)
    fields = [a for a in dir(result) if not a.startswith("_") and not callable(getattr(result, a))]
    print(f"=== consensus on {entity} ===")
    for f in fields[:6]:
        val = getattr(result, f)
        print(f"  {f:18s}: {str(val)[:80]}")
    print()

## 7. Visualization — Conflict Density per Entity

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

if conflicts:
    entity_counts = Counter(getattr(c, "entity", str(c)[:40]) for c in conflicts)
    top = entity_counts.most_common(10)
    if top:
        names, counts = zip(*top)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(range(len(names)), counts, color="#c14a4a")
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.set_xlabel("# cross-document conflicts")
        ax.set_title("Top entities by cross-document conflict count")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("(no conflicts detected — stores agree)")

## 8. Take-Aways

1. **One entity, N stores, one consensus question.** The multi-document picture is "do the sources agree about MDL-001's risk rating?" — a question the harness can answer via geometric tension across stores.
2. **Conflict detection is mechanical.** `find_conflicts()` runs `tension_energy` between corresponding embeddings; high tension = the two stores disagree about the entity. No human review required to flag the candidates.
3. **`ConsensusOracle` is the release-gate hook.** Wire it into a `CompositeOracle` (NB 15) for multi-source release decisions: passing the bench corpus is not enough; the agent's claims must also be consistent with all attested sources.
4. **Stores must share entity vocabulary.** If `q1` says "MDL-001" and `q2` says "MDL_001" (typo), the analyzer will treat them as separate entities. Normalize at ingest (see `feedback_ingest_canonicalize_entities`).
5. **Pair with `CrossStoreCalibration`.** When the stores were ingested at different times with different calibration artifacts, train a cross-store calibrator first (the resolved blocker in project memory `project_cross_store_calibration_blocker`).

## 9. Next Steps

- **NB 19** — `19_release_gates.ipynb` — wire the consensus oracle into the release-gate audit (Ch19)

For theory: `gmsh_testing_ext/chapters/ch18.tex`.